# SWE and SME from scratch — symbolic walkthrough

Derivation of the Shallow-Water Equations (SWE, level=0) and the Shallow
Moment Equations (SME, level>0) directly from incompressible Navier-Stokes.

Everything is mutation-based:

* ``model.apply(op)``  — mutates every equation of the system, returns ``self``.
* ``model.<eq>.apply(op).simplify()`` — chainable in-place mutation on a
  single equation through the proxy.
* ``model.<eq>.solve_for(var)``  — returns an Expression that ``apply``
  consumes directly as ``{var: solution}``.
* ``model.<eq>.remove()`` — drops an equation from the system.

No ``model.equations[name] = model.equations[name].apply(...)`` pattern.
No ``DepthIntegrate`` / ``HydrostaticPressure`` / ``ApplyKinematicBCs`` /
``StressFreeSurface`` / ``ZeroAtmosphericPressure`` / ``SimplifyIntegrals``
shortcuts — everything goes through ``Integrate`` and substitution dicts.

## Imports

In [1]:
import sympy as sp

from zoomy_core.model.models.ins_generator import (
    StateSpace, FullINS, Integrate, Newtonian,
)
from zoomy_core.model.models.sme_model import hydrostatic_scaling

## Step 1 — Start from the raw Navier-Stokes system

In [2]:
state = StateSpace(dimension=2)          # (t, x, z)
model = FullINS(state)
model.describe()

**INS** (continuity, x_momentum, z_momentum)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial z} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial x} u^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial x} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)} + \frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$

**z_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} w{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial z} w^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{zx}{\left(t,x,z \right)} + \frac{\partial}{\partial z} \tau_{zz}{\left(t,x,z \right)}}{\rho}}_{stress} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$


## Step 2 — Hydrostatic assumption on z-momentum

$w = 0$, $\tau_{zz} = \tau_{xz} = \tau_{zx} = 0$ inside z-momentum only.
Chained: ``apply(...)``→proxy, ``.simplify()``→proxy.

In [3]:
model.z_momentum.apply(hydrostatic_scaling(state)).simplify()
model.z_momentum.describe()

**z_momentum** (2 terms)

$$
\begin{aligned}
  & \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$

## Step 3 — Integrate z-momentum analytically to get $p(z)$

Integrate $g + \partial_z p / \rho = 0$ from the current depth $z$ up to the
free surface $\eta$.  `method="analytical"` runs ``sympy.integrate`` on the
whole expression (needed for partial / running integrals).

In [4]:
model.z_momentum.apply(
    Integrate(state.z, state.z, state.eta, method="analytical")
)
model.z_momentum.describe()

**z_momentum** (5 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho} + \frac{p{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)}}{\rho}}_{pressure} \\
  & \underbrace{- g z + g \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right)}_{source}
  &= 0
\end{aligned}
$$

## Step 4 — Atmospheric-pressure BC at the free surface

$p(\eta) = 0$ (atmospheric gauge).  Plain substitution dict.

In [5]:
model.z_momentum.apply({state.p.subs(state.z, state.eta): 0}).simplify()
model.z_momentum.describe()

**z_momentum** (4 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- g z + g b{\left(t,x \right)} + g h{\left(t,x \right)}}_{source}
  &= 0
\end{aligned}
$$

## Step 5 — Substitute the solved $p$ into x-momentum, then drop z-momentum

In [6]:
model.x_momentum.apply(model.z_momentum.solve_for(state.p)).simplify()
model.z_momentum.remove()
model.describe()

**INS** (continuity, x_momentum)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{2 u{\left(t,x,z \right)} \frac{\partial}{\partial x} u{\left(t,x,z \right)} + u{\left(t,x,z \right)} \frac{\partial}{\partial z} w{\left(t,x,z \right)} + w{\left(t,x,z \right)} \frac{\partial}{\partial z} u{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)}}{\rho} - \frac{\frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$


## Step 6 — Newtonian constitutive model

System-level mutation: ``model.apply(op)`` runs the op on every equation
of the system, returns ``self``, so ``.simplify()`` chains.
The internal simplify is linearity-only (``d(b+h)/dx → db/dx + dh/dx``)
and intentionally does **not** chain-rule-expand conservative forms like
``∂_x(u²)``, so they survive for the Leibniz integration in Step 7.

In [7]:
model.apply(Newtonian(state)).simplify()
model.describe()

**INS** (continuity, x_momentum)

**Assumptions:** Newtonian

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{2 u{\left(t,x,z \right)} \frac{\partial}{\partial x} u{\left(t,x,z \right)} + u{\left(t,x,z \right)} \frac{\partial}{\partial z} w{\left(t,x,z \right)} + w{\left(t,x,z \right)} \frac{\partial}{\partial z} u{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)} - \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)} - \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}}_{stress}
  &= 0
\end{aligned}
$$


## Step 7 — Depth-integrate continuity and x-momentum from $b$ to $\eta$

One ``Integrate`` call at system level — per-term auto dispatch picks
Leibniz for $\partial_x$ and the fundamental theorem for $\partial_z$.

In [8]:
model.apply(Integrate(state.z, state.b, state.eta, method="auto"))
model.continuity.describe()

**continuity** (5 terms)

$$
- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz + \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} = 0
$$

In [9]:
model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{\partial}{\partial t} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial t} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. 2 u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. 2 u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz + 2 \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - 2 \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }}}_{convection} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 8 — Resolve $w$ boundary terms via the kinematic BCs

Two substitutions for the $w$ evaluations at bottom and surface.  System-level
``apply(dict)`` applies the dict to every equation and chains into ``simplify``.

In [10]:
u_at_b = state.u.subs(state.z, state.b)
u_at_eta = state.u.subs(state.z, state.eta)
kinematic_bcs = {
    state.w.subs(state.z, state.b):
        sp.Derivative(state.b, state.t) + u_at_b * sp.Derivative(state.b, state.x),
    state.w.subs(state.z, state.eta):
        sp.Derivative(state.eta, state.t) + u_at_eta * sp.Derivative(state.eta, state.x),
}
model.apply(kinematic_bcs).simplify()
model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 9 — Zero tangential stress at surface and bottom

Stress-free surface ($\tau_{xz}|_\eta = 0$) and zero tangential normal
stress at both boundaries ($\tau_{xx}|_b = \tau_{xx}|_\eta = 0$).

In [11]:
stress_free_surface = {state.tau["xz"].subs(state.z, state.eta): 0}
no_tangential_normal_stress = {
    state.tau["xx"].subs(state.z, state.b): 0,
    state.tau["xx"].subs(state.z, state.eta): 0,
}
model.apply(stress_free_surface).apply(no_tangential_normal_stress).simplify()
model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 10 — Bottom stress closure (Navier slip)

$\tau_{xz}|_b = \rho\,(\lambda/\tau_c)\,u|_b$.  Dict + chain.

In [12]:
lamda = sp.Symbol("lamda", positive=True)
tau_c = sp.Symbol("tau_c", positive=True)
friction_closure = {
    state.tau["xz"].subs(state.z, state.b): state.rho * (lamda / tau_c) * u_at_b,
}
model.apply(friction_closure).simplify()
model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## What's left on the board

At this point every equation is:

* Depth-integrated over $[b,\eta]$.
* Closed for $w$, the tangential normal stress, and bottom shear.
* In terms of the velocity field $u(t,x,z)$, its surface/bottom evaluations
  $u|_b$, $u|_\eta$, and volume integrals $\int_b^\eta f\,dz$.

The remaining step is projection against a vertical basis:

* **SWE (level=0)** — constant vertical profile.  Substitute
  ``u(t,x,z) → u_mean(t,x)`` (both in the volume integrals and in the
  surface/bottom evaluations) and evaluate the resulting integrals.
* **SME (level≥1)** — expand ``u(t,x,z) = Σ α_k(t,x) φ_k(ζ)`` and Galerkin-
  test against each ``φ_l``.  Today this is ``Expression.project_onto_basis``;
  it rewrites ``Integral`` nodes and we complement it with explicit
  ``{u|_b: Σ α_k φ_k(0), u|_η: Σ α_k φ_k(1)}`` substitutions for the
  boundary evaluations.

Both projections follow the same mutation pattern used above:
``model.apply(...)`` + chained ``.simplify()``.